# Synthea Explorations

This notebook explores synthetic patient data generated by [Synthea](https://github.com/synthetichealth/synthea).
Synthea outputs FHIR R4 Bundles containing Patient, Practitioner, Encounter, Condition, and other resources.

In [ ]:
import os

from dicom_py_mock_server.services.synthea_utils import SyntheaUtils

SYNTHEA_OUTPUT_FOLDER = "../tests/synthea/fhir"
MAX_BUNDLES = 25


# Check if the output folder exists
if not os.path.exists(SYNTHEA_OUTPUT_FOLDER):
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' does not exist.")
else:
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' found.")

# Load the Synthea output data, parse them into a list of FHIR Bundle dicts, and return the list

syn = SyntheaUtils(synthea_output_folder=SYNTHEA_OUTPUT_FOLDER)

In [ ]:
from typing import List

from dicom_py_mock_server.models.fhir_models import (
    FhirOrganization,
    FhirPractitioner,
    format_human_name_to_dicom,
)

hospitals: List[FhirOrganization] = syn.get_hospital_information()
org: FhirOrganization
practitioners: List[FhirPractitioner]
p: FhirPractitioner
practitioners = syn.get_practitioner_information()
if not hospitals:
    print("Failed to get hospitals")
if not practitioners:
    print("Failed to get practitioners")
for org in hospitals:
    # print(f"{org.id}: {org.name} <- {org.partOf}")
    pass
for p in practitioners:
    name = format_human_name_to_dicom(p.name[0])
    ident = p.identifier[0].value
    print(f"{name} NPI: {ident}")

In [27]:
from typing import List

from fhir.resources.R4B.attachment import Attachment
from fhir.resources.R4B.diagnosticreport import DiagnosticReport
from fhir.resources.R4B.patient import Patient
from fhir.resources.R4B.procedure import Procedure

bundles = syn.scan_synthea_output(max_count=MAX_BUNDLES)

print(f"Loaded {len(bundles)} FHIR Bundles from Synthea output.")

for bundle in bundles:
    patient: Patient = syn.get_patient(bundle)
    dicom_name = syn.get_patient_name(patient)
    patient_id = syn.get_patient_id(patient)
    sex = syn.get_patient_sex(patient)
    birth_date = syn.get_date_time(patient.birthDate)
    print("\n====================================")
    print(f"Patient: DICOM Name: {dicom_name}, Gender: {sex}, ID: {patient_id}, Birth Date: {birth_date[0]}")
    # Imaging Studies
    imagings = syn.get_imaging_studies(bundle)
    for imaging in imagings:
        for s in imaging.series:
            modality = s.modality.display
            print("-- Imaging --")
            print(f"Modality: {modality}")
        enc_ref = imaging.encounter.reference
        encounters = syn.get_encounters(bundle=bundle, encounter_ref=enc_ref)
        for enc in encounters:
            reason = ""
            for reason_code in enc.reasonCode:
                display = reason_code.coding[0].display
            print(f"Encounter On:{enc.period.start}, Ref: {enc_ref}, Reason Code: {reason_code}")
        diags: List[DiagnosticReport] = syn.get_diagnostic_reports(bundle=bundle, encounter_ref=enc_ref)
        procedures: List[Procedure] = syn.get_procedures(bundle=bundle, encounter_ref=enc_ref)
        diag: DiagnosticReport
        proc: Procedure
        for diag in diags:
            if diag and diag.presentedForm:
                for form in diag.presentedForm:
                    match form:
                        case Attachment() as a:
                            text = a.data.decode("utf-8")
                            print(f"Diagnostic Report: {text[0:100]}")
        for proc in procedures:
            code = proc.code.text
            reason = ""
            if proc.reasonReference:
                reason = proc.reasonReference[0].display
            location = proc.location.display
            print(f"Procedure: {code}, {reason} at {location}")

Loaded 3 FHIR Bundles from Synthea output.

Patient: DICOM Name: Hettinger594^Alden634^Frankie174^Mr.^JD, Gender: M, ID: S99916806, Birth Date: 19260821
-- Imaging --
Modality: Computed Radiography
Encounter On:1995-01-21 15:46:29-08:00, Ref: urn:uuid:1bbc0b16-2965-fc1c-b8e2-01ed9b1da080, Reason Code: fhir_comments=None extension=None id=None coding=[Coding(fhir_comments=None, extension=None, id=None, code='274531002', code__ext=None, display='Abnormal findings diagnostic imaging heart+coronary circulat (finding)', display__ext=None, system='http://snomed.info/sct', system__ext=None, userSelected=None, userSelected__ext=None, version=None, version__ext=None)] text=None text__ext=None
Diagnostic Report: 
1995-01-21

# Chief Complaint
- Thirst
- Frequent Urination
- Tingling in Hands and Feet
- urinary 
Procedure: Pre-surgery testing (procedure),  at Fitchburg Outpatient Clinic
Procedure: Blood group typing (procedure),  at Fitchburg Outpatient Clinic
Procedure: Electrocardiographic proc